# Projeto Aplicado — Sprint 1 · Base LLVE (R05)
**Segmentação de Clientes por Valor (RFM e Clusterização) para Otimização de Marketing e Retenção em E-commerce de Moda**

| R05 | Extrair, pseudonimizar e limpar a base da LLVE |
|---|---|
| Critério de aceite | Base LLVE tratada, com os compradores válidos e **sem dados pessoais** |

> **LGPD:** este notebook lê o arquivo bruto da pasta `DADOS LLVE` (privada, fora do GitHub) e grava apenas a versão pseudonimizada em `PROCESSADOS`. Nenhuma célula exibe nome, e-mail, telefone, documento ou endereço. **Não salve este notebook no GitHub.**

### 5.1 Ambiente e parâmetros

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

from pathlib import Path
import hashlib, numpy as np, pandas as pd

BASE_DIR    = Path('/content/drive/MyDrive/PROJETO APLICADO CD/PROJETO APLICADO')
LLVE_DIR    = BASE_DIR / 'DADOS LLVE'                 # dados brutos — acesso restrito
ARQ_LLVE    = LLVE_DIR / 'clientes_llve_pa.csv'
OUTPUT_DIR  = BASE_DIR / 'PROCESSADOS'
FIG_DIR     = BASE_DIR / 'FIGURAS'
SEED = 42
PERCENTIL_CORTE = 0.999                              # mesma regra D4 da Olist, calculada na base da LLVE

pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
print('Arquivo encontrado:', ARQ_LLVE.exists())

### 5.2 Chave secreta da pseudonimização
O identificador do cliente será um **hash SHA-256 do e-mail combinado com uma chave secreta**. Sem a chave, não é possível voltar ao e-mail; com ela, só você consegue reverter o código na Sprint 3 para montar os públicos.

**Como criar a chave (uma vez só):** no Colab, clique no ícone de chave 🔑 na barra lateral (*Secrets*) → *Adicionar novo secret* → nome `LLVE_SALT`, valor = uma frase longa e aleatória que só você conhece → ative *Acesso ao notebook*. A chave não fica gravada no notebook.

In [ ]:
SALT = userdata.get('LLVE_SALT')
assert SALT and len(SALT) >= 16, 'Crie o secret LLVE_SALT com pelo menos 16 caracteres'

def pseudonimizar(valor):
    """Hash SHA-256 (chave + e-mail normalizado), truncado em 16 caracteres."""
    if pd.isna(valor) or str(valor).strip() == '':
        return np.nan
    texto = SALT + str(valor).strip().lower()
    return hashlib.sha256(texto.encode('utf-8')).hexdigest()[:16]

print('Chave carregada · teste de hash:', pseudonimizar('exemplo@teste.com'))

### 5.3 Leitura do arquivo e inspeção da estrutura (sem exibir dados)
A célula abaixo mostra **apenas** nomes de colunas, tipos, % de nulos e nº de valores distintos.

In [ ]:
def ler_csv(caminho):
    for enc in ('utf-8', 'utf-8-sig', 'latin-1'):
        try:
            return pd.read_csv(caminho, sep=None, engine='python', encoding=enc), enc
        except UnicodeDecodeError:
            continue
    raise ValueError('Não foi possível ler o arquivo')

bruto, encoding = ler_csv(ARQ_LLVE)
print(f'{len(bruto):,} linhas × {bruto.shape[1]} colunas · encoding {encoding}')

estrutura = pd.DataFrame({
    'coluna': bruto.columns,
    'tipo': bruto.dtypes.astype(str).values,
    '% nulos': (bruto.isna().mean() * 100).round(1).values,
    'valores distintos': bruto.nunique().values,
})
estrutura

### 5.4 Pseudonimização e seleção de colunas
Decisões (registradas para o relatório):
| # | Decisão | Justificativa |
|---|---|---|
| L1 | `id_cliente` = hash SHA-256 (chave secreta + e-mail) | E-mail é único por cliente (1.879 valores distintos) e é a chave usada pelo Meta Ads para públicos personalizados |
| L2 | Excluir nome, CPF/CNPJ, telefone, endereço, número, complemento, bairro e CEP | Dados pessoais desnecessários para o RFM (minimização — LGPD) |
| L3 | Excluir gênero, nascimento, aniversário e tags | 100% nulos |
| L4 | Manter cidade, UF, total consumido, nº de compras, última compra, data de cadastro e consentimento de marketing | Variáveis do RFM, da H4 e do playbook (consentimento define quem pode receber e-mail/WhatsApp) |

In [ ]:
MAPA_COLUNAS = {
    'Cidade': 'cidade',
    'Estado': 'uf',
    'Total Consumido (BRL)': 'total_consumido',
    'Número de Compras': 'num_compras',
    'Última Compra': 'ultima_compra',
    'Data': 'data_cadastro',
    'Marketing': 'aceita_marketing',
}

llve = bruto[list(MAPA_COLUNAS)].rename(columns=MAPA_COLUNAS)
llve.insert(0, 'id_cliente', bruto['E-mail'].map(pseudonimizar))

assert llve['id_cliente'].notna().all(), 'há clientes sem e-mail'
assert llve['id_cliente'].is_unique, 'colisão ou e-mail duplicado'

# remove a base bruta da memória da sessão
colunas_excluidas = [c for c in bruto.columns if c not in MAPA_COLUNAS]
del bruto
print(f'{len(llve):,} clientes pseudonimizados · colunas excluídas ({len(colunas_excluidas)}):', ', '.join(colunas_excluidas))

### 5.5 Ajuste de tipos

In [ ]:
for col in ['ultima_compra', 'data_cadastro']:
    convertido = pd.to_datetime(llve[col], dayfirst=True, errors='coerce')
    falhas = convertido.isna().sum() - llve[col].isna().sum()
    print(f'{col:<15} datas convertidas: {convertido.notna().sum():,} · não reconhecidas: {falhas}')
    llve[col] = convertido

llve['total_consumido'] = pd.to_numeric(llve['total_consumido'], errors='coerce')
llve['num_compras'] = llve['num_compras'].astype('int64')
llve['uf'] = llve['uf'].str.strip().str.upper().fillna('Não informado')
llve['aceita_marketing'] = llve['aceita_marketing'].astype(str).str.strip()
llve.dtypes.to_frame('tipo')

### 5.6 Qualidade antes do tratamento

In [ ]:
def retrato(df, etapa):
    return {'etapa': etapa, 'clientes': len(df), 'compradores': (df['num_compras'] > 0).sum(),
            'receita_R$': df['total_consumido'].sum()}
log_llve = [retrato(llve, '0. Base pseudonimizada')]

consistencia = pd.DataFrame([
    {'verificação': 'cadastros sem compra (num_compras = 0)',         'clientes': (llve['num_compras'] == 0).sum()},
    {'verificação': 'compra registrada mas total consumido = 0',      'clientes': ((llve['num_compras'] > 0) & (llve['total_consumido'] <= 0)).sum()},
    {'verificação': 'total consumido > 0 sem compra registrada',      'clientes': ((llve['num_compras'] == 0) & (llve['total_consumido'] > 0)).sum()},
    {'verificação': 'comprador sem data de última compra',            'clientes': ((llve['num_compras'] > 0) & llve['ultima_compra'].isna()).sum()},
    {'verificação': 'última compra anterior ao cadastro',             'clientes': (llve['ultima_compra'] < llve['data_cadastro'].dt.normalize()).sum()},
    {'verificação': 'UF não informada',                               'clientes': (llve['uf'] == 'Não informado').sum()},
])
consistencia['% da base'] = consistencia['clientes'] / len(llve) * 100
consistencia

### 5.7 Remoção dos cadastros sem compra (L5)
Cadastros que nunca compraram (leads frios ou espúrios, identificados na Imersão Profunda) não têm Recência, Frequência nem Valor e não entram na segmentação.

In [ ]:
compradores = llve[(llve['num_compras'] > 0) & (llve['total_consumido'] > 0)].copy()
log_llve.append(retrato(compradores, '1. Remoção dos cadastros sem compra'))
print(f'Removidos: {len(llve) - len(compradores):,} · compradores: {len(compradores):,}')

### 5.8 Valores extremos (L6 — equivalente à D4 da Olist)
Como a base é por cliente, o valor por pedido é aproximado pelo **ticket médio** (total consumido ÷ nº de compras). O corte no P99,9 é calculado sobre a própria distribuição da LLVE, e não reaproveita o valor de R$ 2.331 da Olist.

In [ ]:
compradores['ticket_medio'] = compradores['total_consumido'] / compradores['num_compras']
corte_llve = compradores['ticket_medio'].quantile(PERCENTIL_CORTE)
extremos = compradores['ticket_medio'] > corte_llve

print(f'Corte P{PERCENTIL_CORTE*100:g}: R$ {corte_llve:,.2f} · clientes acima: {extremos.sum()} · '
      f'{compradores.loc[extremos, "total_consumido"].sum() / compradores["total_consumido"].sum():.1%} da receita')
compradores = compradores[~extremos].copy()
log_llve.append(retrato(compradores, f'2. Remoção de clientes acima do P{PERCENTIL_CORTE*100:g} (ticket médio)'))

### 5.9 Qualidade depois do tratamento e validação com a Imersão Profunda
Conferência com os números do relatório (seção 1.1.1): 1.318 compradores, 79,1% com uma única compra, ticket médio de R$ 413 (mediana R$ 331) e 71 clientes no perfil ideal (2+ compras e ticket médio acima de R$ 500).

In [ ]:
funil_llve = pd.DataFrame(log_llve)
funil_llve['% receita retida'] = funil_llve['receita_R$'] / funil_llve['receita_R$'].iloc[0] * 100
display(funil_llve)

validacao = pd.Series({
    'compradores': len(compradores),
    '% com uma única compra': (compradores['num_compras'] == 1).mean() * 100,
    '% recompradores': (compradores['num_compras'] > 1).mean() * 100,
    'ticket médio (R$)': compradores['ticket_medio'].mean(),
    'ticket mediano (R$)': compradores['ticket_medio'].median(),
    'clientes no perfil ideal (2+ compras e ticket > R$ 500)': ((compradores['num_compras'] >= 2) & (compradores['ticket_medio'] > 500)).sum(),
    'última compra mais recente': compradores['ultima_compra'].max().strftime('%d/%m/%Y'),
    'última compra mais antiga': compradores['ultima_compra'].min().strftime('%d/%m/%Y'),
    '% com UF informada': (compradores['uf'] != 'Não informado').mean() * 100,
    '% que aceita marketing': (compradores['aceita_marketing'].str.lower().isin(['sim', 'yes', 'true', '1'])).mean() * 100,
}).to_frame('valor')
validacao

### 5.10 Exportação da base pseudonimizada

In [ ]:
COLUNAS_PII = {'Nome completo', 'CPF/CNPJ', 'E-mail', 'Telefone de Contato', 'Endereço', 'Número',
               'Complemento', 'Bairro', 'CEP'}
assert not COLUNAS_PII.intersection(compradores.columns), 'há colunas com dados pessoais na base de saída'

arq_llve = OUTPUT_DIR / 'llve_clientes_tratados.parquet'
compradores.to_parquet(arq_llve, index=False)
print(f'Salvo: {arq_llve.name} · {len(compradores):,} clientes × {compradores.shape[1]} colunas · sem dados pessoais')

### 5.11 Checklist e limitações do R05
- [x] Chave secreta fora do notebook (Colab Secrets) e `id_cliente` pseudonimizado
- [x] Dados pessoais excluídos e base bruta removida da memória (L1–L3)
- [x] Cadastros sem compra removidos (L5) e valores extremos tratados (L6)
- [x] Conferência com os números da Imersão Profunda
- [x] Base salva em `PROCESSADOS/llve_clientes_tratados.parquet`, verificada sem dados pessoais

**Limitações (base por cliente, não por pedido):** (i) não é possível excluir pedidos cancelados caso estejam somados no total consumido; (ii) o corte de valores extremos usa o ticket médio como aproximação do valor por pedido; (iii) cerca de 45% dos cadastros não têm UF, o que restringe o teste da H4 na LLVE. Se a exportação de pedidos da Nuvemshop for obtida, as etapas 5.4 a 5.8 serão refeitas no nível do pedido, com as mesmas regras da Olist.